# 12 — Voice and Multimodal Agents

        **Mode:** `live`  
        **Version:** Praval `0.8.0`  
        **Course link:** New in Praval 0.8

        This notebook makes the framework's execution visible. It records actual
        stages and renders the messages or runtime events that connect them.

        **You will see**

        - A committed WAV passes through real STT.
- An agent response becomes real synthesized speech and is transcribed again.
- The same agent inspects a real image input.

        Run the cells from top to bottom. Committed notebooks contain no saved
        output, so everything shown is produced by your installed Praval package.


In [ ]:
from __future__ import annotations

import html as _html
import json
import os
import time
from pathlib import Path

from IPython.display import HTML, display

os.environ.setdefault("PRAVAL_OBSERVABILITY", "off")


from praval import get_provider_registry
from praval.models import ModelResponse, ProviderCapabilities


class NotebookLifecycleProvider:
    """Credential-free adapter for agents whose handlers do not call a model."""

    provider_name = "notebook-lifecycle"
    capabilities = ProviderCapabilities(text=True)

    def __init__(self, config):
        self.config = config

    def invoke(self, request, tools=None):
        return ModelResponse(
            content="Notebook lifecycle response",
            provider=self.provider_name,
            model=request.model,
        )

    def close(self):
        return None


_notebook_registry = get_provider_registry()
_notebook_registry.register_provider(
    "notebook-lifecycle",
    NotebookLifecycleProvider,
    default_model="notebook-lifecycle-model",
    capabilities=NotebookLifecycleProvider.capabilities,
)
os.environ.setdefault("PRAVAL_DEFAULT_PROVIDER", "notebook-lifecycle")
os.environ.setdefault("PRAVAL_DEFAULT_MODEL", "notebook-lifecycle-model")

EVENTS = []
_STARTED = time.perf_counter()


def stage(actor, action, detail="", spore=None):
    """Capture one observable execution stage."""
    EVENTS.append(
        {
            "ms": round((time.perf_counter() - _STARTED) * 1000, 1),
            "actor": actor,
            "action": action,
            "detail": detail,
            "spore_id": getattr(spore, "id", "") if spore else "",
        }
    )


def show_flow(*steps):
    cards = []
    for index, (name, detail) in enumerate(steps):
        if index:
            cards.append('<div style="font-size:24px;color:#557">→</div>')
        cards.append(
            '<div style="padding:12px 16px;border:1px solid #b8c7dc;'
            'border-radius:12px;background:#f7fbff;min-width:130px">'
            f'<b>{_html.escape(name)}</b><br>'
            f'<span style="color:#456;font-size:12px">'
            f'{_html.escape(detail)}</span></div>'
        )
    display(
        HTML(
            '<div style="display:flex;gap:10px;align-items:center;'
            'flex-wrap:wrap;margin:12px 0">' + "".join(cards) + "</div>"
        )
    )


def show_timeline(events=None):
    events = EVENTS if events is None else events
    rows = []
    for item in events:
        rows.append(
            "<tr>"
            f"<td>{item['ms']:.1f}</td>"
            f"<td><b>{_html.escape(str(item['actor']))}</b></td>"
            f"<td>{_html.escape(str(item['action']))}</td>"
            f"<td>{_html.escape(str(item['detail']))}</td>"
            f"<td><code>{_html.escape(str(item['spore_id'])[:12])}</code></td>"
            "</tr>"
        )
    display(
        HTML(
            '<table style="border-collapse:collapse;width:100%">'
            '<thead><tr><th>ms</th><th>actor</th><th>stage</th>'
            '<th>detail</th><th>spore</th></tr></thead><tbody>'
            + "".join(rows)
            + "</tbody></table>"
        )
    )


def show_spore(spore, label="Spore on the Reef"):
    payload = json.loads(spore.to_json())
    rendered = _html.escape(json.dumps(payload, indent=2, sort_keys=True))
    display(
        HTML(
            '<div style="border-left:5px solid #7b61ff;padding:10px 14px;'
            'background:#faf9ff;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def show_json(value, label="Result"):
    rendered = _html.escape(json.dumps(value, indent=2, sort_keys=True, default=str))
    display(
        HTML(
            '<div style="border-left:5px solid #18a999;padding:10px 14px;'
            'background:#f5fffd;border-radius:8px">'
            f'<b>{_html.escape(label)}</b><pre style="white-space:pre-wrap">'
            f"{rendered}</pre></div>"
        )
    )


def require_env(*names):
    missing = [name for name in names if not os.environ.get(name)]
    if missing:
        raise RuntimeError("Missing required notebook configuration: " + ", ".join(missing))
    return {name: os.environ[name] for name in names}


def find_example_asset(relative):
    relative = Path(relative)
    starts = [Path.cwd(), *Path.cwd().parents]
    for root in starts:
        for candidate in (root / relative, root / "examples" / relative):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"Could not locate example asset: {relative}")


## Live OpenAI prerequisites

This is the visual counterpart to the release voice certificate. It performs
`WAV → STT → Agent → TTS → WAV → STT` and also sends a committed red PNG to
the configured multimodal model. No local speech substitute is accepted.


In [ ]:
import base64
import gzip
import tempfile
import wave

from IPython.display import Audio, Image
from praval import Agent, ContentPart

values = require_env(
    "OPENAI_API_KEY",
    "PRAVAL_OPENAI_MODEL",
    "PRAVAL_OPENAI_TRANSCRIPTION_MODEL",
    "PRAVAL_OPENAI_TTS_MODEL",
    "PRAVAL_OPENAI_TTS_VOICE",
)
workspace = tempfile.TemporaryDirectory(prefix="praval-voice-notebook-")
output = Path(workspace.name)


def wav_metadata(path):
    with wave.open(str(path), "rb") as stream:
        frames = stream.getnframes()
        rate = stream.getframerate()
        return {
            "channels": stream.getnchannels(),
            "sample_rate": rate,
            "frames": frames,
            "duration_seconds": round(frames / rate, 3),
        }


def normalized_words(value):
    clean = "".join(char.lower() if char.isalnum() else " " for char in value)
    return set(clean.split())


In [ ]:
voice_fixture = find_example_asset(
    "certification/assets/voice_input.wav.gz.base64"
)
input_audio = gzip.decompress(base64.b64decode(voice_fixture.read_text()))
input_path = output / "voice-input.wav"
input_path.write_bytes(input_audio)
input_info = wav_metadata(input_path)
assert input_info["frames"] > 0
stage("audio", "fixture loaded", f"{input_info['duration_seconds']} seconds")
display(Audio(filename=str(input_path)))
show_json(input_info, "Input WAV")


In [ ]:
agent = Agent(
    "course-voice-agent",
    provider="openai",
    model=values["PRAVAL_OPENAI_MODEL"],
    config={"temperature": 0, "max_output_tokens": 64, "timeout": 60},
)
transcript = agent.transcribe(
    input_path,
    model=values["PRAVAL_OPENAI_TRANSCRIPTION_MODEL"],
    language="en",
    timeout=60,
)
required = {"praval", "speech", "transcription", "synthesis"}
assert required <= normalized_words(transcript)
stage("OpenAI STT", "transcribed", transcript)

response = agent.generate(
    "The user said: " + transcript +
    " Reply with exactly: Praval voice round trip succeeded."
)
assert {"praval", "voice", "succeeded"} <= normalized_words(response.content)
stage("agent", "response generated", response.content)


In [ ]:
reply_audio = agent.speak(
    response.content,
    model=values["PRAVAL_OPENAI_TTS_MODEL"],
    voice=values["PRAVAL_OPENAI_TTS_VOICE"],
    response_format="wav",
    timeout=60,
)
reply_path = output / "voice-reply.wav"
reply_path.write_bytes(reply_audio)
reply_info = wav_metadata(reply_path)
assert reply_info["frames"] > 0
stage("OpenAI TTS", "speech generated", f"{reply_info['duration_seconds']} seconds")
display(Audio(filename=str(reply_path)))

roundtrip = agent.transcribe(
    reply_path,
    model=values["PRAVAL_OPENAI_TRANSCRIPTION_MODEL"],
    language="en",
    timeout=60,
)
assert {"praval", "voice", "succeeded"} <= normalized_words(roundtrip)
stage("OpenAI STT", "round-trip verified", roundtrip)
show_json(
    {
        "input_transcript": transcript,
        "agent_reply": response.content,
        "roundtrip_transcript": roundtrip,
        "reply_wav": reply_info,
    },
    "Real voice round trip",
)


In [ ]:
image_fixture = find_example_asset("certification/assets/image_input.png.base64")
image_bytes = base64.b64decode(image_fixture.read_text())
image_path = output / "red.png"
image_path.write_bytes(image_bytes)
display(Image(filename=str(image_path), width=96, height=96))

image_response = agent.generate(
    [
        ContentPart.text_part("Name the dominant color in this image."),
        ContentPart.image_base64(
            base64.b64encode(image_bytes).decode("ascii"), "image/png"
        ),
    ]
)
assert "red" in image_response.content.lower()
stage("multimodal model", "image inspected", image_response.content)
show_timeline()


In [ ]:
agent.close()
workspace.cleanup()
stage("agent", "closed", "temporary media removed")
